## Loading the data
Load raw posts as .csv from Azure blob storage.

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
import pandas as pd
from openai import OpenAI
import numpy as np

openai_api_key = os.getenv("OPENAI_KEY") # Your API key here

def load_embeddings_numpy(filepath="embeddings.npy"):
    """Loads embeddings from a NumPy array."""
    return np.load(filepath)

dataset_name = "bills-hoyle-2022" # "bills-hoyle-2022" or "wikitext-103-hoyle-2022"
train_data = pd.read_json(f"../data/{dataset_name}/train.metadata.jsonl", lines=True)
#train_embeddings = load_embeddings_numpy(f"data/{dataset_name}/train_5k_sample_embeddings.npy")
test_data = pd.read_json(f"../data/{dataset_name}/test.metadata.jsonl", lines=True)
#test_embeddings = load_embeddings_numpy(f"data/{dataset_name}/test_embeddings.npy")

In [ ]:
#train_data = train_data[:80]
#test_data = test_data[:20]

In [ ]:
%matplotlib inline

In [ ]:
import logging
for handler in logging.root.handlers[:]:
   logging.root.removeHandler(handler)
# logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                   level=logging.NOTSET,
                   filename='../data/logs/bt-logs.log')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_keywords_from_df(df, text_col="tokenized_text", topic_col="supercategory", top_n=10):
    """
    Extracts top N keywords for each topic using TF-IDF.

    Args:
        df (pd.DataFrame): DataFrame with a text column and a topic label column.
        text_col (str): Name of the column containing document text.
        topic_col (str): Name of the column with topic labels.
        top_n (int): Number of keywords per topic.

    Returns:
        dict: {topic: [keyword1, keyword2, ...]} with top keywords for each topic.
    """
    keywords_per_topic = {}

    for topic, group in df.groupby(topic_col):
        texts = group[text_col].astype(str).tolist()  # Ensure it's a list of strings
        vectorizer = TfidfVectorizer(stop_words='english', max_features=10_000)
        tfidf_matrix = vectorizer.fit_transform(texts)

        feature_names = vectorizer.get_feature_names_out()
        tfidf_scores = tfidf_matrix.mean(axis=0).A1  # Average TF-IDF per word

        top_keywords = [feature_names[i] for i in tfidf_scores.argsort()[-top_n:][::-1]]
        keywords_per_topic[topic] = top_keywords

    return keywords_per_topic

## Apply pre-processing
Note, first the contents are still full text and the mean length is in chars, after preprocessing it into bag of words it is a list of strings and length is in word count therefor

In [ ]:
input_column = "summary"
gold_label = "topic"

train_docs = train_data[input_column].tolist()
test_docs = test_data[input_column].tolist()

print(len(train_docs))
print(len(test_docs))
print(train_docs[0][:200])

In [ ]:
# Precalculate embeddings
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # or any
train_embeddings = embedding_model.encode(train_docs)
test_embeddings = embedding_model.encode(test_docs)

# Training the model: LDA

In [ ]:
run = "paper-bm-bertopic-v1-bills"

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:8080")

In [ ]:
def get_or_create_experiment(experiment_name):
    """
    Retrieve the ID of an existing MLflow experiment or create a new one if it doesn't exist.

    This function checks if an experiment with the given name exists within MLflow.
    If it does, the function returns its ID. If not, it creates a new experiment
    with the provided name and returns its ID.

    Parameters:
    - experiment_name (str): Name of the MLflow experiment.

    Returns:
    - str: ID of the existing or newly created MLflow experiment.
    """

    if experiment := mlflow.get_experiment_by_name(experiment_name):
        return experiment.experiment_id
    else:
        return mlflow.create_experiment(experiment_name)

In [ ]:
llmbt_experiment_id = get_or_create_experiment("ICNLSP Experiment BERTopic")
llmbt_experiment_id

In [ ]:
from typing import List
import json

from openai import OpenAI
from textonomy.utils import metrics
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import os
import mlflow
import mlflow.data
os.environ["TOKENIZERS_PARALLELISM"] = "true"

mlflow.openai.autolog()

client = OpenAI(api_key=openai_api_key)  # Make sure openai_api_key is defined

# Run the LLM+BERTopic model
training_info_tag = "first try: first run with bills"

model_name = "bertopic"
printprefix = str(run) + " - " + model_name + ": "


ks = [25,23,22]

for i, k in enumerate(ks):

    model_name = "bertopic"
    printprefix = str(i) + " - " + model_name + ": "
    training_info_tag = "bills attempt #1"
    with mlflow.start_run(experiment_id=llmbt_experiment_id):
        # Set a tag that we can use to remind ourselves what this run was for
        mlflow.set_tag(
            "Training Info", training_info_tag
        )
        mlflow.log_param("model_name", model_name)

        dataset = mlflow.data.from_pandas(train_data, source="Azure Blob Storage", name=dataset_name)
        mlflow.log_input(dataset, context=dataset_name)

        model_params = {
        "nr_topics": k,
        }

        print(printprefix + "befor bert count vectorizer")
        
        vectorizer_model = CountVectorizer(stop_words="english")

        print(printprefix + "after bert count vectorizer")

        model = BERTopic(nr_topics=model_params["nr_topics"], vectorizer_model=vectorizer_model)
        
        print(printprefix + "after bert init")
        
        model_params = model.get_params()

        print(printprefix + "after model_params")


        mlflow.log_params(model_params)

        # vectorizer_model = CountVectorizer(
        #     ngram_range=params["ngram_range"], stop_words=params["stop_words"]
        # )

        print(printprefix + "model fitting")

        _ = model.fit_transform(train_docs, embeddings=train_embeddings)

        print(printprefix + "model fitting done")
        print(printprefix + "Calculating topics for test data")

        test_data['predicted_label'], _ = model.transform(test_docs, embeddings=test_embeddings)
        #topic_probs = get_document_vectors(model, test_corpus, model_params["k"])
        topics = test_data['predicted_label'].to_list()

        print(printprefix + "Calculating metrics")

        purpose = "A taxonomy to organise these summaries of US congressional bills into (main) topics. Aim at around 15-25 topics. Examples: 'Government Operations', 'Environment'..."
        topk = 10

        topic_words = extract_keywords_from_df(test_data, text_col='tokenized_text', topic_col='predicted_label', top_n=10)
        mlflow.log_dict(topic_words, "tf_idf_topic_words.json")
        topic_words_list = [i for i in topic_words.values()]
        ref_corpus_texts = [doc.split() for doc in test_data['tokenized_text'].to_list()]

        metric_dict, llm_usefulness_reasoning = metrics.evaluate_bt_model(topic_model=model, topics=topics, purpose=purpose, openai=client, topk=topk, preprocessed_docs=ref_corpus_texts, model_labels=topics, gold_labels=test_data[gold_label].to_list(), embeddings=test_embeddings, topic_words_list=topic_words_list)
        
        # Log metrics
        mlflow.log_param("eval_topk", topk)

        metric_dict['outlier_ratio'] = topics.count(-1)/len(topics)
        metric_dict['num_train_docs'] = train_data.id.nunique()
        metric_dict['num_test_docs'] = test_data.id.nunique()
        metric_dict['num_topics'] = model_params["nr_topics"]
        
        valid_metrics, invalid_metrics = {}, {}
        for k, v in metric_dict.items():
            try:
                valid_metrics[k] = float(v)
            except (ValueError, TypeError):
                invalid_metrics[k] = v  # Store invalid values separately
        # Log valid metrics
        mlflow.log_metrics(valid_metrics)
        # If there are invalid metrics, save them to a text file and log as an artifact
        if invalid_metrics:
            invalid_metrics_path = "../data/tmp/invalid_metrics.txt"
            with open(invalid_metrics_path, "w") as f:
                json.dump(invalid_metrics, f, indent=4)  # Save as JSON for readability
            mlflow.log_artifact(invalid_metrics_path)  # Log the file in MLflow
            print(printprefix + f"Invalid metrics saved to {invalid_metrics_path}")
            os.remove(invalid_metrics_path)  # Remove the local file
        
        mlflow.log_text(purpose, "user_defined_purpose_for_U_LLM_USEFULNESS.txt")
        mlflow.log_text(str(llm_usefulness_reasoning), "U_LLM_USEFULNESS_reasoning.txt")

        # Print metrics
        #metric_df = create_metrics_dataframe(metric_dict)
        metric_df = pd.DataFrame.from_dict(metric_dict, orient="index", columns=["value"])
        display(metric_df)

        # Infer signature based on the data and model
        #signature = infer_signature(train_docs, model.transform(train_docs))

        # Log the model with signature
        #model_info = mlflow.sklearn.log_model(model, "lda_model", signature=signature)

        print("Logged run '" + str(run) + "' to MLflow.")
        #print(f"Model URI: {model_info.model_uri}")
        print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")